In [1]:
import xarray as xr
import numpy as np
import pandas as pd

In [2]:
casename = 'EEP_MITgcm185Lvgrid_Whitt2026hgrid'
path = '/glade/derecho/scratch/iranjan/archive/weekly_eep/ocn/hist/'
files = casename + '.mom6.h.z.****-**-***.nc'

In [3]:
ds = xr.open_mfdataset(path+files,drop_variables=["average_DT",])

In [4]:
# My case didn't have volume transports in z-output. Calculate from mass transports
ds['uh'] = ds['umo']/1035.   # kg /s to m^3 /s
ds['vh'] = ds['vmo']/1035. 

# Fix the annoying difference in variable names for the vertical coordinate
ds = ds.rename({'z_l':'zl','z_i':'zi'})

In [5]:
ds['true_w'] = ds['vert_remap_h_tendency'].cumsum(dim='zl')

In [6]:
print(ds)

<xarray.Dataset> Size: 39GB
Dimensions:                         (time: 25, zl: 38, yh: 288, xq: 901,
                                     yq: 289, xh: 900, zi: 39, nbnd: 2)
Coordinates:
  * xq                              (xq) float64 7kB 190.0 190.1 ... 264.9 265.0
  * yh                              (yh) float64 2kB -11.96 -11.88 ... 11.96
  * zl                              (zl) float64 304B 2.5 10.0 ... 7.5e+03 8e+03
  * zi                              (zi) float64 312B 0.0 5.0 ... 8.25e+03
  * time                            (time) datetime64[ns] 200B 2015-01-02T12:...
  * nbnd                            (nbnd) float64 16B 1.0 2.0
  * xh                              (xh) float64 7kB 190.0 190.1 ... 264.9 265.0
  * yq                              (yq) float64 2kB -12.0 -11.92 ... 11.92 12.0
Data variables: (12/42)
    uo                              (time, zl, yh, xq) float32 986MB dask.array<chunksize=(1, 38, 288, 901), meta=np.ndarray>
    vo                              (time, z

In [7]:
import numpy as np
import xarray as xr
 
 
def compute_true_w_profile(
    ds, center_lat, center_lon, box_size,
    start_depth, end_depth, depth_step,
    var='true_w', area_weights=None,
):
    """
    Compute a spatial-box-averaged depth profile of true w from a MOM6
    history dataset, on the SAME schema compute_w_planefit's w_est uses:
    depth negative-down, time = ds's own time coordinate directly (no
    relabeling needed now that obs generation and model output timestamps
    line up exactly with no gaps).
 
    Parameters
    ----------
    ds : xr.Dataset
        MOM6 history dataset with `var` on dims (time, zl, yh, xh).
    center_lat, center_lon : float
        Center of the averaging box (0-360 longitude convention, matching
        ds.xh).
    box_size : float
        Full width/height of the box in degrees.
    start_depth, end_depth, depth_step : float
        Define the MIDPOINT depths conceptually (e.g. 8, 80, 2). Output
        depths are the corresponding INTERFACE depths (same convention
        compute_w_planefit uses for w_est's depth coordinate).
    var : str
        Name of the true-w variable in ds. Default 'true_w'.
    area_weights : xr.DataArray or None
        Optional (yh, xh) cell-area weights for a proper area-weighted mean.
        If None, an UNWEIGHTED mean over the box is used — fine for a
        small, roughly-uniform-area box.
 
    Returns
    -------
    xr.DataArray, dims (time, depth), name f'{var}_mean'.
    depth is NEGATIVE-down (matches w_est directly).
    time is ds's own time coordinate, unmodified.
    """
    lat_min, lat_max = center_lat - box_size / 2, center_lat + box_size / 2
    lon_min, lon_max = center_lon - box_size / 2, center_lon + box_size / 2
 
    box = ds[var].sel(yh=slice(lat_min, lat_max), xh=slice(lon_min, lon_max))
    if box.sizes.get('yh', 0) == 0 or box.sizes.get('xh', 0) == 0:
        raise ValueError(
            f"Box selection (lat {lat_min}-{lat_max}, lon {lon_min}-{lon_max}) "
            f"returned no grid points — check center_lat/center_lon are in "
            f"the same convention as ds.yh/ds.xh (e.g. 0-360 longitude, "
            f"not -180/180)."
        )
 
    if area_weights is not None:
        w = area_weights.sel(yh=slice(lat_min, lat_max), xh=slice(lon_min, lon_max))
        box_mean = box.weighted(w).mean(dim=['yh', 'xh'])
    else:
        box_mean = box.mean(dim=['yh', 'xh'])
 
    # --- interface-depth grid, following compute_w_planefit's own
    # z_top = obs_z[0] + dz/2 convention, reproduced so the two grids are
    # numerically identical, not just visually similar.
    n_mid = int(round((end_depth - start_depth) / depth_step)) + 1
    z_top = start_depth - depth_step / 2
    interp_depths_posdown = z_top + np.arange(n_mid + 1) * depth_step
 
    zl = box_mean.zl.values
    if interp_depths_posdown.max() > zl.max() or interp_depths_posdown.min() < zl.min():
        print(f"WARNING: requested depth range ({interp_depths_posdown.min()}-"
              f"{interp_depths_posdown.max()} m) extends beyond ds's zl range "
              f"({zl.min()}-{zl.max()} m). Out-of-range points will be NaN.")
 
    box_mean_np = box_mean.transpose('time', 'zl').values  # (ntime, nzl)
    out_vals = np.full((box_mean_np.shape[0], len(interp_depths_posdown)), np.nan)
    for i in range(box_mean_np.shape[0]):
        out_vals[i] = np.interp(
            interp_depths_posdown, zl, box_mean_np[i],
            left=np.nan, right=np.nan,
        )
 
    # NEGATIVE-down, to match w_est's depth coordinate directly.
    # time = ds's own time coordinate, kept as-is (no relabeling) since it
    # now lines up exactly with the obs generation schedule.
    out = xr.DataArray(
        out_vals, dims=('time', 'depth'),
        coords={'time': box_mean.time, 'depth': -interp_depths_posdown},
        name=f'{var}_mean',
    )
    out.attrs['center_lat'] = center_lat
    out.attrs['center_lon'] = center_lon
    out.attrs['box_size'] = box_size
    out.attrs['depth_convention'] = 'negative-down, interface levels (matches w_est)'
 
    out.to_netcdf("true_w.nc")
    return out

In [8]:
true_w = compute_true_w_profile(ds, 0.5, 220, 1, 8, 80, 2)

In [9]:
print(true_w)

<xarray.DataArray 'true_w_mean' (time: 25, depth: 38)> Size: 8kB
array([[-1.16065734e-05, -1.44402955e-05, -1.69117740e-05,
        -1.90210090e-05, -2.11302440e-05, -2.32394790e-05,
        -2.53487140e-05, -2.75049448e-05, -2.97081714e-05,
        -3.19113980e-05, -3.41146247e-05, -3.63178513e-05,
        -3.85210779e-05, -4.04031786e-05, -4.13219015e-05,
        -4.22406245e-05, -4.31593474e-05, -4.40780704e-05,
        -4.49967933e-05, -4.59155162e-05, -4.68342392e-05,
        -4.77529621e-05, -4.86716851e-05, -4.85878216e-05,
        -4.83607314e-05, -4.81336413e-05, -4.79065511e-05,
        -4.76794610e-05, -4.74523709e-05, -4.72252807e-05,
        -4.69981906e-05, -4.67711004e-05, -4.65440103e-05,
        -4.63169201e-05, -4.60898300e-05, -4.47066373e-05,
        -4.33234446e-05, -4.19402518e-05],
       [-6.54540554e-06, -7.92905166e-06, -9.01115282e-06,
        -9.79170900e-06, -1.05722652e-05, -1.13528214e-05,
        -1.21333775e-05, -1.26460799e-05, -1.28909284e-05,
       

In [10]:
print(true_w.time.values[:3])

['2015-01-02T12:00:00.000000000' '2015-01-09T12:00:00.000000000'
 '2015-01-16T12:00:00.000000000']


In [11]:
print(ds.time.values[:3])

['2015-01-02T12:00:00.000000000' '2015-01-09T12:00:00.000000000'
 '2015-01-16T12:00:00.000000000']


In [12]:
diffs = pd.Series(ds.time.values).diff().dropna()
print(diffs.unique())  # should show a single value: 7 days

<TimedeltaArray>
['7 days']
Length: 1, dtype: timedelta64[ns]
